# Final SGCC LightGBM Research Model Preparation

This notebook prepares a reproducible **research model bundle** from the SGCC customer-level electricity-theft dataset. It does not establish readiness for Nigerian prepaid-token or vending fraud detection.

## What this notebook does
1. Loads `data set.csv` from your original `archive.zip`.
2. Rebuilds the established customer-level features.
3. Fits the selected LightGBM configuration on all available labelled SGCC customers.
4. Saves the model, feature-engineering code, feature schema, package versions, metadata, and nested-CV evaluation summary into `sgcc_final_model_bundle.zip`.

**Important evaluation caveat:** the model will be fitted on all available rows. Do not calculate or report training-set predictions as performance. The nested cross-validation estimates (outer PR-AUC 0.4344, ROC-AUC 0.8279, F1 0.4304) are the current development estimates. The provisional threshold 0.5602 is the median of the three thresholds selected inside the nested CV training folds; it is not a production-calibrated threshold.


## 1. Install dependencies and upload the original dataset

In [ ]:
!pip -q install lightgbm scikit-learn joblib pandas numpy

from google.colab import files
import io, zipfile, warnings, os, json, shutil, importlib.metadata
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import sklearn, lightgbm, joblib
from lightgbm import LGBMClassifier
warnings.filterwarnings('ignore')

uploaded = files.upload()  # Upload archive.zip only
zip_name = next((n for n in uploaded if n.lower().endswith('.zip')), None)
if zip_name is None:
    raise ValueError('Please upload the original archive.zip dataset.')
with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as z:
    csv_name = next((n for n in z.namelist() if n.lower().endswith('data set.csv')), None)
    if csv_name is None:
        raise ValueError('Could not find data set.csv inside the ZIP.')
    raw = pd.read_csv(z.open(csv_name))

required = {'CONS_NO', 'FLAG'}
if not required.issubset(raw.columns):
    raise ValueError(f'Missing required columns: {required - set(raw.columns)}')
if raw['CONS_NO'].isna().any() or raw['FLAG'].isna().any():
    raise ValueError('Identifier or target contains missing values; inspect the source data.')
if not set(pd.Series(raw['FLAG']).astype(int).unique()).issubset({0,1}):
    raise ValueError('FLAG must contain binary 0/1 labels.')
print('Raw dataset shape:', raw.shape)
print('Unique customers:', raw['CONS_NO'].nunique())
print('Positive-label rate:', round(raw['FLAG'].astype(int).mean(), 5))


Saving archive.zip to archive.zip
Raw dataset shape: (42372, 1036)
Unique customers: 42372
Positive-label rate: 0.08532


## 2. Rebuild the established feature pipeline

In [ ]:
def build_features(df):
    id_col, target = 'CONS_NO', 'FLAG'
    date_cols = [c for c in df.columns if c not in (id_col, target)]
    bad_dates = [c for c in date_cols if df[c].isna().mean() > 0.99]
    use_dates = [c for c in date_cols if c not in bad_dates]
    dates = pd.to_datetime(use_dates, format='%m/%d/%Y', errors='coerce')
    X = df[use_dates].to_numpy(dtype=np.float32, copy=True)
    y = df[target].astype(np.int8).to_numpy()
    n, p = X.shape
    missing = np.isnan(X); obs = ~missing
    cnt = obs.sum(axis=1).astype(np.float32)
    zero = obs & (X == 0); positive = obs & (X > 0)
    feats = {}
    def add(k, v): feats[k] = np.asarray(v, dtype=np.float32)
    add('observed_days', cnt)
    add('missing_pct', missing.mean(axis=1) * 100)
    add('zero_pct_observed', np.divide(zero.sum(axis=1), cnt, out=np.zeros(n, dtype=np.float32), where=cnt>0)*100)
    add('zero_days', zero.sum(axis=1)); add('nonzero_days', positive.sum(axis=1))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        add('mean_daily', np.nanmean(X, axis=1)); add('std_daily', np.nanstd(X, axis=1))
        mx = np.full(n, np.nan, dtype=np.float32)
        valid_rows = cnt > 0
        mx[valid_rows] = np.nanmax(X[valid_rows], axis=1)
        add('max_daily_log', np.log1p(mx))
    qs = [10,25,50,75,90,95,99]
    quant = {q: np.full(n, np.nan, dtype=np.float32) for q in qs}
    for lo in range(0, n, 1500):
        hi = min(lo+1500, n); block = np.sort(X[lo:hi], axis=1); counts = cnt[lo:hi].astype(np.int64)
        rr = np.arange(hi-lo)
        for q in qs:
            pos = (counts-1)*(q/100.0); left=np.floor(np.maximum(pos,0)).astype(int); right=np.ceil(np.maximum(pos,0)).astype(int); frac=pos-left
            vals = block[rr,left] + (block[rr,right]-block[rr,left])*frac
            vals[counts==0] = np.nan; quant[q][lo:hi] = vals
    for q,v in quant.items(): add('median_daily' if q==50 else f'q{q}_daily',v)
    wins_mean=np.full(n,np.nan,dtype=np.float32); wins_std=np.full(n,np.nan,dtype=np.float32)
    for lo in range(0,n,1500):
        hi=min(lo+1500,n); block=X[lo:hi]; cap=quant[99][lo:hi,None]
        with np.errstate(invalid='ignore'):
            clipped=np.minimum(block,cap)
            wins_mean[lo:hi]=np.nanmean(clipped,axis=1); wins_std[lo:hi]=np.nanstd(clipped,axis=1)
    add('mean_winsor99',wins_mean); add('std_winsor99',wins_std)
    add('cv_daily',feats['std_daily']/(feats['mean_daily']+1e-3))
    longz=np.zeros(n,dtype=np.int16); curz=np.zeros(n,dtype=np.int16); longm=np.zeros(n,dtype=np.int16); curm=np.zeros(n,dtype=np.int16)
    for j in range(p):
        curz=np.where(zero[:,j],curz+1,0); longz=np.maximum(longz,curz)
        curm=np.where(missing[:,j],curm+1,0); longm=np.maximum(longm,curm)
    add('longest_zero_streak',longz); add('longest_missing_streak',longm)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore',RuntimeWarning)
        def nanmean(a): return np.nanmean(a,axis=1)
        windows={'first_30d':(0,min(30,p)),'last_30d':(max(0,p-30),p),'last_90d':(max(0,p-90),p),'last_180d':(max(0,p-180),p),'first_half':(0,p//2),'second_half':(p//2,p),'prior_90d':(max(0,p-120),max(0,p-30))}
        for name,(a,b) in windows.items():
            add(name+'_mean',nanmean(X[:,a:b]))
            denom=obs[:,a:b].sum(axis=1)
            add(name+'_zero_pct',np.divide(zero[:,a:b].sum(axis=1),denom,out=np.zeros(n,dtype=np.float32),where=denom>0)*100)
        add('half_mean_change',feats['second_half_mean']-feats['first_half_mean'])
        add('half_mean_change_ratio',(feats['second_half_mean']-feats['first_half_mean'])/(np.abs(feats['first_half_mean'])+1e-3))
        add('last30_vs_prior90_ratio',(feats['last_30d_mean']-feats['prior_90d_mean'])/(np.abs(feats['prior_90d_mean'])+1e-3))
        for k,a in enumerate(range(0,p,90),1): add(f'block{k:02d}_mean',nanmean(X[:,a:min(a+90,p)]))
        t=np.arange(p,dtype=np.float64)[None,:]; valid=obs.astype(np.float64); vals=np.nan_to_num(X,nan=0).astype(np.float64)
        cntd=valid.sum(axis=1); sumt=(valid*t).sum(axis=1); sumx=vals.sum(axis=1); sumtx=(vals*t).sum(axis=1); sumt2=(valid*t*t).sum(axis=1)
        den=cntd*sumt2-sumt*sumt
        slope=np.divide(cntd*sumtx-sumt*sumx,den,out=np.zeros(n),where=den!=0)
        add('linear_trend_slope',slope); add('linear_trend_slope_scaled',slope/(feats['mean_daily']+1e-3))
        weekdays=np.array([d.weekday() for d in dates]); wkday=np.isin(weekdays,[0,1,2,3,4]); wkend=~wkday
        add('weekday_mean',nanmean(X[:,wkday])); add('weekend_mean',nanmean(X[:,wkend]))
        add('weekend_weekday_ratio',(feats['weekend_mean']-feats['weekday_mean'])/(np.abs(feats['weekday_mean'])+1e-3))
        months=np.array([d.month for d in dates])
        for m in range(1,13):
            mask=months==m
            if mask.any(): add(f'month_{m:02d}_mean',nanmean(X[:,mask]))
    F=pd.DataFrame(feats).replace([np.inf,-np.inf],np.nan)
    F.insert(0,'FLAG',y); F.insert(0,'CONS_NO',df[id_col].astype(str).values)
    return F, bad_dates

features, excluded_dates = build_features(raw)
feature_columns = [c for c in features.columns if c not in ("CONS_NO", "FLAG")]
X = features[feature_columns].copy()
y = features["FLAG"].astype(int).to_numpy()

print("Feature table shape:", features.shape)
print("Engineered feature count:", len(feature_columns))
print("Excluded near-all-missing date columns:", excluded_dates)
print("Positive-label rate:", round(float(y.mean()), 5))
if X.columns.duplicated().any():
    raise ValueError("Duplicate feature names detected.")
arr = X.to_numpy(dtype=float, copy=False)
if np.isinf(arr).any():
    raise ValueError("Unexpected infinite values in feature matrix.")


Feature table shape: (42372, 68)
Engineered feature count: 66
Excluded near-all-missing date columns: ['10/3/2014']
Positive-label rate: 0.08532


## 3. Fit the selected LightGBM configuration on all labelled customers

In [ ]:
# Established baseline selected during development.
training_params = {
    'objective': 'binary', 'class_weight': 'balanced', 'n_estimators': 650,
    'learning_rate': 0.03, 'num_leaves': 31, 'colsample_bytree': 0.85,
    'subsample': 0.85, 'subsample_freq': 1, 'reg_lambda': 2,
    'min_child_samples': 30, 'n_jobs': -1, 'random_state': 2026,
    'verbosity': -1,
}
final_model = LGBMClassifier(**training_params)
final_model.fit(X, y)
print('Final research model fitted on all rows:', len(y))
print('Model feature count:', final_model.n_features_in_)
print('Feature count matches:', int(final_model.n_features_in_) == len(feature_columns))
# Do NOT compute training-set F1/PR-AUC and present it as generalization performance.


Final research model fitted on all rows: 42372
Model feature count: 66
Feature count matches: True


## 4. Record nested cross-validation results and provisional threshold

In [ ]:
# Values transcribed from the completed nested cross-validation output.
outer_fold_results = pd.DataFrame([
    {'outer_fold': 1, 'selected_model': 'LightGBM', 'threshold_from_inner_oof': 0.589044, 'inner_selected_model_PR_AUC': 0.4146, 'outer_PR_AUC': 0.4362, 'outer_ROC_AUC': 0.8265, 'outer_precision': 0.4016, 'outer_recall': 0.4672, 'outer_F1': 0.4319, 'outer_specificity': 0.9351, 'TP': 563, 'FP': 839, 'FN': 642, 'TN': 12080, 'outer_test_customers': 14124},
    {'outer_fold': 2, 'selected_model': 'LightGBM', 'threshold_from_inner_oof': 0.549292, 'inner_selected_model_PR_AUC': 0.4140, 'outer_PR_AUC': 0.4503, 'outer_ROC_AUC': 0.8333, 'outer_precision': 0.3738, 'outer_recall': 0.5212, 'outer_F1': 0.4354, 'outer_specificity': 0.9186, 'TP': 628, 'FP': 1052, 'FN': 577, 'TN': 11867, 'outer_test_customers': 14124},
    {'outer_fold': 3, 'selected_model': 'LightGBM', 'threshold_from_inner_oof': 0.560202, 'inner_selected_model_PR_AUC': 0.4268, 'outer_PR_AUC': 0.4167, 'outer_ROC_AUC': 0.8238, 'outer_precision': 0.3779, 'outer_recall': 0.4830, 'outer_F1': 0.4240, 'outer_specificity': 0.9258, 'TP': 582, 'FP': 958, 'FN': 623, 'TN': 11961, 'outer_test_customers': 14124},
])
outer_fold_results.to_csv('nested_cv_outer_fold_results.csv', index=False)
outer_summary = pd.DataFrame([
    {'metric':'outer_PR_AUC','mean':0.4344,'sd_across_outer_folds':0.0169},
    {'metric':'outer_ROC_AUC','mean':0.8279,'sd_across_outer_folds':0.0049},
    {'metric':'outer_precision','mean':0.3844,'sd_across_outer_folds':0.0150},
    {'metric':'outer_recall','mean':0.4905,'sd_across_outer_folds':0.0277},
    {'metric':'outer_F1','mean':0.4304,'sd_across_outer_folds':0.0058},
    {'metric':'outer_specificity','mean':0.9265,'sd_across_outer_folds':0.0083},
])
outer_summary.to_csv('nested_cv_summary.csv', index=False)
provisional_threshold = float(np.median([0.589044, 0.549292, 0.560202]))
print('Provisional threshold (median of inner-only thresholds):', round(provisional_threshold, 6))
display(outer_summary)


Provisional threshold (median of inner-only thresholds): 0.560202


,metric,mean,sd_across_outer_folds
0,outer_PR_AUC,0.4344,0.0169
1,outer_ROC_AUC,0.8279,0.0049
2,outer_precision,0.3844,0.0150
3,outer_recall,0.4905,0.0277
4,outer_F1,0.4304,0.0058
5,outer_specificity,0.9265,0.0083


## 5. Package the model and everything needed to reproduce feature generation

In [ ]:
import importlib.metadata

out_dir = 'sgcc_final_model_bundle'
os.makedirs(out_dir, exist_ok=True)
joblib.dump(final_model, os.path.join(out_dir, 'lightgbm_sgcc_research.joblib'), compress=3)

# Save the exact feature-generation function as a standalone Python module.
feature_source = "def build_features(df):\n    id_col, target = 'CONS_NO', 'FLAG'\n    date_cols = [c for c in df.columns if c not in (id_col, target)]\n    bad_dates = [c for c in date_cols if df[c].isna().mean() > 0.99]\n    use_dates = [c for c in date_cols if c not in bad_dates]\n    dates = pd.to_datetime(use_dates, format='%m/%d/%Y', errors='coerce')\n    X = df[use_dates].to_numpy(dtype=np.float32, copy=True)\n    y = df[target].astype(np.int8).to_numpy()\n    n, p = X.shape\n    missing = np.isnan(X); obs = ~missing\n    cnt = obs.sum(axis=1).astype(np.float32)\n    zero = obs & (X == 0); positive = obs & (X > 0)\n    feats = {}\n    def add(k, v): feats[k] = np.asarray(v, dtype=np.float32)\n    add('observed_days', cnt)\n    add('missing_pct', missing.mean(axis=1) * 100)\n    add('zero_pct_observed', np.divide(zero.sum(axis=1), cnt, out=np.zeros(n, dtype=np.float32), where=cnt>0)*100)\n    add('zero_days', zero.sum(axis=1)); add('nonzero_days', positive.sum(axis=1))\n    with warnings.catch_warnings():\n        warnings.simplefilter('ignore', RuntimeWarning)\n        add('mean_daily', np.nanmean(X, axis=1)); add('std_daily', np.nanstd(X, axis=1))\n        mx = np.full(n, np.nan, dtype=np.float32)\n        valid_rows = cnt > 0\n        mx[valid_rows] = np.nanmax(X[valid_rows], axis=1)\n        add('max_daily_log', np.log1p(mx))\n    qs = [10,25,50,75,90,95,99]\n    quant = {q: np.full(n, np.nan, dtype=np.float32) for q in qs}\n    for lo in range(0, n, 1500):\n        hi = min(lo+1500, n); block = np.sort(X[lo:hi], axis=1); counts = cnt[lo:hi].astype(np.int64)\n        rr = np.arange(hi-lo)\n        for q in qs:\n            pos = (counts-1)*(q/100.0); left=np.floor(np.maximum(pos,0)).astype(int); right=np.ceil(np.maximum(pos,0)).astype(int); frac=pos-left\n            vals = block[rr,left] + (block[rr,right]-block[rr,left])*frac\n            vals[counts==0] = np.nan; quant[q][lo:hi] = vals\n    for q,v in quant.items(): add('median_daily' if q==50 else f'q{q}_daily',v)\n    wins_mean=np.full(n,np.nan,dtype=np.float32); wins_std=np.full(n,np.nan,dtype=np.float32)\n    for lo in range(0,n,1500):\n        hi=min(lo+1500,n); block=X[lo:hi]; cap=quant[99][lo:hi,None]\n        with np.errstate(invalid='ignore'):\n            clipped=np.minimum(block,cap)\n            wins_mean[lo:hi]=np.nanmean(clipped,axis=1); wins_std[lo:hi]=np.nanstd(clipped,axis=1)\n    add('mean_winsor99',wins_mean); add('std_winsor99',wins_std)\n    add('cv_daily',feats['std_daily']/(feats['mean_daily']+1e-3))\n    longz=np.zeros(n,dtype=np.int16); curz=np.zeros(n,dtype=np.int16); longm=np.zeros(n,dtype=np.int16); curm=np.zeros(n,dtype=np.int16)\n    for j in range(p):\n        curz=np.where(zero[:,j],curz+1,0); longz=np.maximum(longz,curz)\n        curm=np.where(missing[:,j],curm+1,0); longm=np.maximum(longm,curm)\n    add('longest_zero_streak',longz); add('longest_missing_streak',longm)\n    with warnings.catch_warnings():\n        warnings.simplefilter('ignore',RuntimeWarning)\n        def nanmean(a): return np.nanmean(a,axis=1)\n        windows={'first_30d':(0,min(30,p)),'last_30d':(max(0,p-30),p),'last_90d':(max(0,p-90),p),'last_180d':(max(0,p-180),p),'first_half':(0,p//2),'second_half':(p//2,p),'prior_90d':(max(0,p-120),max(0,p-30))}\n        for name,(a,b) in windows.items():\n            add(name+'_mean',nanmean(X[:,a:b]))\n            denom=obs[:,a:b].sum(axis=1)\n            add(name+'_zero_pct',np.divide(zero[:,a:b].sum(axis=1),denom,out=np.zeros(n,dtype=np.float32),where=denom>0)*100)\n        add('half_mean_change',feats['second_half_mean']-feats['first_half_mean'])\n        add('half_mean_change_ratio',(feats['second_half_mean']-feats['first_half_mean'])/(np.abs(feats['first_half_mean'])+1e-3))\n        add('last30_vs_prior90_ratio',(feats['last_30d_mean']-feats['prior_90d_mean'])/(np.abs(feats['prior_90d_mean'])+1e-3))\n        for k,a in enumerate(range(0,p,90),1): add(f'block{k:02d}_mean',nanmean(X[:,a:min(a+90,p)]))\n        t=np.arange(p,dtype=np.float64)[None,:]; valid=obs.astype(np.float64); vals=np.nan_to_num(X,nan=0).astype(np.float64)\n        cntd=valid.sum(axis=1); sumt=(valid*t).sum(axis=1); sumx=vals.sum(axis=1); sumtx=(vals*t).sum(axis=1); sumt2=(valid*t*t).sum(axis=1)\n        den=cntd*sumt2-sumt*sumt\n        slope=np.divide(cntd*sumtx-sumt*sumx,den,out=np.zeros(n),where=den!=0)\n        add('linear_trend_slope',slope); add('linear_trend_slope_scaled',slope/(feats['mean_daily']+1e-3))\n        weekdays=np.array([d.weekday() for d in dates]); wkday=np.isin(weekdays,[0,1,2,3,4]); wkend=~wkday\n        add('weekday_mean',nanmean(X[:,wkday])); add('weekend_mean',nanmean(X[:,wkend]))\n        add('weekend_weekday_ratio',(feats['weekend_mean']-feats['weekday_mean'])/(np.abs(feats['weekday_mean'])+1e-3))\n        months=np.array([d.month for d in dates])\n        for m in range(1,13):\n            mask=months==m\n            if mask.any(): add(f'month_{m:02d}_mean',nanmean(X[:,mask]))\n    F=pd.DataFrame(feats).replace([np.inf,-np.inf],np.nan)\n    F.insert(0,'FLAG',y); F.insert(0,'CONS_NO',df[id_col].astype(str).values)\n    return F, bad_dates"
with open(os.path.join(out_dir, 'feature_engineering.py'), 'w', encoding='utf-8') as f:
    f.write('import warnings\nimport numpy as np\nimport pandas as pd\n\n')
    f.write(feature_source + '\n')
with open(os.path.join(out_dir, 'feature_columns.json'), 'w', encoding='utf-8') as f:
    json.dump(feature_columns, f, indent=2)

packages = {}
for package in ['numpy','pandas','scikit-learn','lightgbm','joblib']:
    try: packages[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: packages[package] = None
metadata = {
    'model_name': 'LightGBM SGCC customer-level electricity-theft research model',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'dataset_file': 'data set.csv from archive.zip',
    'n_customers': int(len(raw)),
    'n_unique_customer_ids': int(raw['CONS_NO'].nunique()),
    'positive_rate': float(y.mean()),
    'target_column': 'FLAG', 'identifier_column': 'CONS_NO',
    'feature_count': int(len(feature_columns)),
    'feature_columns_file': 'feature_columns.json',
    'feature_engineering_source': 'feature_engineering.py',
    'excluded_near_all_missing_dates': list(excluded_dates),
    'training_parameters': training_params,
    'provisional_threshold': provisional_threshold,
    'threshold_note': 'Median of three thresholds selected inside nested-CV training folds. Provisional only; not calibrated for production or Nigerian operations.',
    'nested_cv_outer_summary': {'PR_AUC_mean':0.4344,'PR_AUC_sd':0.0169,'ROC_AUC_mean':0.8279,'ROC_AUC_sd':0.0049,'precision_mean':0.3844,'recall_mean':0.4905,'F1_mean':0.4304,'specificity_mean':0.9265},
    'nested_cv_pooled': {'PR_AUC':0.4341,'ROC_AUC':0.8279},
    'package_versions': packages,
    'trained_on_all_available_rows': True,
    'training_set_metrics_reported_as_generalization': False,
    'scope_warning': 'SGCC customer-level electricity-theft classification only. Not validated for Nigerian prepaid-token/recharge/vending transaction fraud.',
    'deployment_warning': 'Do not use for enforcement or adverse action without independent local validation, calibrated operating thresholds, human review, and governance.'
}
with open(os.path.join(out_dir, 'model_metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)
outer_fold_results.to_csv(os.path.join(out_dir, 'nested_cv_outer_fold_results.csv'), index=False)
outer_summary.to_csv(os.path.join(out_dir, 'nested_cv_summary.csv'), index=False)
requirements = '\n'.join([f'{k}=={v}' for k,v in packages.items() if v]) + '\n'
with open(os.path.join(out_dir, 'requirements_versions.txt'), 'w', encoding='utf-8') as f: f.write(requirements)
readme = f'''# SGCC final research model bundle\n\nFiles:\n- `lightgbm_sgcc_research.joblib`: fitted LightGBM model.\n- `feature_engineering.py`: feature engineering function used in training.\n- `feature_columns.json`: exact feature order expected by the model.\n- `model_metadata.json`: parameters, versions, provisional threshold and limitations.\n- `nested_cv_outer_fold_results.csv`, `nested_cv_summary.csv`: development evaluation results.\n- `requirements_versions.txt`: observed package versions from training environment.\n\nThis model was fitted on all {len(raw):,} SGCC rows. The fitted model itself has no independent test score. The nested cross-validation summary is the development estimate: PR-AUC 0.4344 (fold SD 0.0169), ROC-AUC 0.8279, F1 0.4304.\n\nThe provisional threshold is {provisional_threshold:.6f}; it is not a production-calibrated threshold. The dataset is for customer-level electricity-theft classification, not Nigerian prepaid token transaction fraud. Independent local validation is required before any operational use.\n'''
with open(os.path.join(out_dir, 'README.md'), 'w', encoding='utf-8') as f: f.write(readme)
zip_path = shutil.make_archive('sgcc_final_model_bundle', 'zip', out_dir)
print('Created:', zip_path)
print('Bundle contents:', sorted(os.listdir(out_dir)))
files.download(zip_path)


Created: /content/sgcc_final_model_bundle.zip
Bundle contents: ['README.md', 'feature_columns.json', 'feature_engineering.py', 'lightgbm_sgcc_research.joblib', 'model_metadata.json', 'nested_cv_outer_fold_results.csv', 'nested_cv_summary.csv', 'requirements_versions.txt']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>